# Alorica → Qualtrics | Post-Call Survey Ingestion Pipeline

**Owner:** Plinary — Disciplined Intelligence / CR&I Team

**Purpose:** Daily ETL pipeline that loads Alorica post-call survey files (one per OpCo),
transforms and validates the data, and uploads completed responses to Qualtrics for each OpCo.

**OpCos covered:** RGE · NYSEG · CMP · SCG · CNG · UIC

---

## How to run
1. Run **Section 0** (all setup cells) once per session
2. Run **Section 1** to load the pre-prepared Alorica CSV files from OneDrive
3. Run **Section 2** to transform and validate
4. Run **Section 3 — Execute** to upload to Qualtrics

> **Extract note (future):** The Extract section (Section 1) currently reads files that have been
> manually downloaded and placed in the OneDrive folders below. Once the Genesys Cloud API
> integration is complete, this section will be replaced by an automated pull from the Alorica
> contact center platform. The Transform and Load sections will remain unchanged.

> **File naming convention:** `ALO {OPCO}_Survey_Data YYYY-MM-DD.csv`
> e.g. `ALO RGE_Survey_Data 2026-10-13.csv`

> **SharePoint note:** If the OneDrive folders are not synced on this machine, the pipeline
> will raise a `FileNotFoundError`. Ensure the relevant OneDrive folders are synced before running.

# 0. Environment Setup

## Library Imports

| Library | Purpose |
|---|---|
| `pandas` | DataFrame operations |
| `datetime` | Date math |
| `requests` | HTTP calls to Qualtrics API |
| `urllib3` | SSL warning suppression |
| `difflib.get_close_matches` | Fuzzy-match column names to Qualtrics fields |
| `glob` | File pattern matching |
| `os` | File path handling |
| `zipfile`, `io` | Unzip Qualtrics export files (duplicate check) |

In [ ]:
import pandas as pd
from datetime import date, timedelta, datetime, timezone
import requests
import urllib3
from difflib import get_close_matches
import glob
import os
import zipfile
import io
import time
requests.packages.urllib3.disable_warnings()

## Qualtrics Credentials

| Variable | Description |
|---|---|
| `API_TOKEN` | Qualtrics API authentication token |
| `DATA_CENTER` | Qualtrics data center identifier |
| `SURVEYS_ID` | Dict mapping each OpCo to its Qualtrics survey ID |

> ⚠️ **Security:** Move credentials to environment variables before sharing or deploying to Azure.

In [ ]:
# ============================================================
# QUALTRICS CONFIGURATION
# ============================================================

API_TOKEN   = "dZZueEgbaPrCuSTXEtIp2sz0EqlJNxg93jYEod7U"
DATA_CENTER = "iad1"

SURVEYS_ID = {
    "RGE"  : "SV_cvxFLJeNk0tQm6q",
    "NYSEG": "SV_5dLTpj4MDbbc1bU",
    "CMP"  : "SV_ebvEco2ILaRjPqS",
    "CNG"  : "SV_cuK9q1z7FFGaiLc",
    "SCG"  : "SV_6fZBPa0TfJq4Xrg",
    "UIC"  : "SV_3R9Pe8Lcy5ZaIQW",
}

## OneDrive Output Folders

Resolves the local OneDrive sync path for each OpCo.

| OpCo | Layout | OneDrive Subfolder |
|---|---|---|
| RGE   | RGE/NYSEG | `Alorica_RGE` |
| NYSEG | RGE/NYSEG | `Alorica_NYSEG` |
| CMP   | UIL Group | `Alorica_UIL` |
| SCG   | UIL Group | `Alorica_UIL` |
| CNG   | UIL Group | `Alorica_UIL` |
| UIC   | UIL Group | `Alorica_UIL` |

> **UIL Group note:** UIC, SCG, CNG, and CMP share the same `Alorica_UIL` folder.
> Each OpCo's files are distinguished by the filename prefix (e.g. `ALO CMP_Survey_Data`).
> UIL is the name of the larger group of companies that includes these four OpCos.

In [ ]:
# ============================================================
# FILE PATHS
# ============================================================

BASE_PATH = os.path.join(
    os.path.expanduser("~"),
    "OneDrive - IBERDROLA S.A",
    "General - Customer Research",
    "Post Call Survey",
    "Post Call Survey Data 2025"
)

OPCO_CONFIG = {
    "RGE": {
        "folder"     : os.path.join(BASE_PATH, "Alorica_RGE"),
        "file_prefix": "ALO RGE_Survey_Data",
        "layout"     : "rge_nyseg",
    },
    "NYSEG": {
        "folder"     : os.path.join(BASE_PATH, "Alorica_NYSEG"),
        "file_prefix": "ALO NYSEG_Survey_Data",
        "layout"     : "rge_nyseg",
    },
    "CMP": {
        "folder"     : os.path.join(BASE_PATH, "Alorica_UIL"),
        "file_prefix": "ALO CMP_Survey_Data",
        "layout"     : "uil",
    },
    "SCG": {
        "folder"     : os.path.join(BASE_PATH, "Alorica_UIL"),
        "file_prefix": "ALO SCG_Survey_Data",
        "layout"     : "uil",
    },
    "CNG": {
        "folder"     : os.path.join(BASE_PATH, "Alorica_UIL"),
        "file_prefix": "ALO CNG_Survey_Data",
        "layout"     : "uil",
    },
    "UIC": {
        "folder"     : os.path.join(BASE_PATH, "Alorica_UIL"),
        "file_prefix": "ALO UIC_Survey_Data",
        "layout"     : "uil",
    },
}

## Helper Functions

- `find_file(opco, date_str)` — locates the daily Alorica CSV/XLSX for a given OpCo and date
- `check_latest_qualtrics_date(opco)` — queries Qualtrics for the most recent Date already ingested (duplicate prevention)

In [ ]:
def find_file(opco, date_str):
    """
    Looks for: <prefix> <YYYY-MM-DD>.csv  or  <prefix> <YYYY-MM-DD>.xlsx
    in the configured OneDrive folder for the given OpCo.
    Returns the full path if found, None otherwise.
    """
    cfg    = OPCO_CONFIG[opco]
    folder = cfg["folder"]
    prefix = cfg["file_prefix"]

    for ext in ["csv", "xlsx"]:
        pattern = os.path.join(folder, f"{prefix} {date_str}.{ext}")
        matches = glob.glob(pattern)
        if matches:
            return matches[0]

    return None


def check_latest_qualtrics_date(opco):
    """
    Queries Qualtrics for the most recent Date already ingested for this OpCo.
    Filters to last 7 days by RecordedDate to avoid downloading full history.
    Returns (latest_date, latest_recorded, batch_count) or None if no recent responses.
    """
    survey_id    = SURVEYS_ID[opco]
    headers      = {"X-API-TOKEN": API_TOKEN, "Content-Type": "application/json"}
    start_filter = (datetime.now(timezone.utc) - timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%SZ")

    # Step 1 — Start export job
    url     = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{survey_id}/export-responses"
    payload = {"format": "csv", "startDate": start_filter}
    r       = requests.post(url, headers=headers, json=payload, verify=False)
    r.raise_for_status()
    progress_id = r.json()["result"]["progressId"]
    print(f"  [{opco}] Qualtrics check started...")

    # Step 2 — Poll until ready
    poll_url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{survey_id}/export-responses/{progress_id}"
    file_id  = None
    for _ in range(20):
        time.sleep(3)
        poll   = requests.get(poll_url, headers={"X-API-TOKEN": API_TOKEN}, verify=False)
        result = poll.json()["result"]
        if result["status"] == "complete":
            file_id = result["fileId"]
            break

    if not file_id:
        print(f"  [{opco}] ⚠ Qualtrics export timed out — skipping duplicate check")
        return None

    # Step 3 — Download and parse
    file_url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{survey_id}/export-responses/{file_id}/file"
    file_r   = requests.get(file_url, headers={"X-API-TOKEN": API_TOKEN}, verify=False)
    z        = zipfile.ZipFile(io.BytesIO(file_r.content))
    df       = pd.read_csv(z.open(z.namelist()[0]), skiprows=[1, 2])

    if df.empty:
        print(f"  [{opco}] No responses in Qualtrics in last 7 days")
        return None

    # Step 4 — Find max Date within the most recent RecordedDate batch
    df["RecordedDate"] = pd.to_datetime(df["RecordedDate"], errors="coerce")
    latest_recorded    = df["RecordedDate"].max().date()
    latest_batch       = df[df["RecordedDate"].dt.date == latest_recorded]

    # Alorica uses a "Date" column (not "Date/Time" like iQor)
    date_col = "Date" if "Date" in df.columns else df.columns[0]
    latest_date = pd.to_datetime(latest_batch[date_col], errors="coerce").max()

    print(f"  [{opco}] Latest in Qualtrics : {latest_date.date()} ({len(latest_batch)} records, ingested {latest_recorded})")
    return latest_date, latest_recorded, len(latest_batch)

---
# 1. Extract

## Extract — Load Alorica Files from OneDrive

**Current approach (manual prep shortcut):**
Files are manually downloaded from the Alorica reporting portal and placed in the
corresponding OneDrive folder before running this pipeline. This section reads them from disk.

**Future state:**
This section will be replaced by an automated extract from the Genesys Cloud / Alorica API
once the `SurveyAsyncAggregationQuery` integration is complete. The Transform and Load
sections below will not need to change.

**File naming convention:**
```
ALO RGE_Survey_Data YYYY-MM-DD.csv
ALO NYSEG_Survey_Data YYYY-MM-DD.csv
ALO CMP_Survey_Data YYYY-MM-DD.csv
ALO SCG_Survey_Data YYYY-MM-DD.csv
ALO CNG_Survey_Data YYYY-MM-DD.csv
ALO UIC_Survey_Data YYYY-MM-DD.csv
```

**Output:**
- `RAW_DATA` — `{"RGE": df, "NYSEG": df, ...}` raw DataFrames, one per OpCo
- `PIPELINE_DATE` — the date string being processed (set once, used throughout)

In [ ]:
# ============================================================
# EXTRACT — Set date and load files
# ============================================================
# Change PIPELINE_DATE to process a specific date, or leave as
# None to default to yesterday.

PIPELINE_DATE = None   # e.g. "2026-10-13"  or  None for yesterday

if PIPELINE_DATE is None:
    PIPELINE_DATE = (datetime.now(timezone.utc) - timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Pipeline date : {PIPELINE_DATE}")
print(f"OpCos         : {', '.join(OPCO_CONFIG.keys())}")
print()

RAW_DATA = {}

for opco in OPCO_CONFIG:
    filepath = find_file(opco, PIPELINE_DATE)
    if not filepath:
        print(f"[{opco}] ⚠  File not found — skipping ({OPCO_CONFIG[opco]['file_prefix']} {PIPELINE_DATE}.csv)")
        continue

    try:
        if filepath.endswith(".xlsx"):
            df = pd.read_excel(filepath)
        else:
            df = pd.read_csv(filepath)

        RAW_DATA[opco] = df
        print(f"[{opco}] ✓ Loaded {len(df)} rows from {os.path.basename(filepath)}")
    except Exception as e:
        print(f"[{opco}] ❌ Read error: {e}")

print(f"\nExtract complete — {len(RAW_DATA)}/{len(OPCO_CONFIG)} OpCos loaded.")

---
# 2. Transform

## Transform — Rename Columns & Standardize Types

**What it does:**
1. Renames source columns to consistent canonical names
2. Casts score columns to numeric (NaN preserved for missing responses)

Two column layouts exist depending on OpCo:

| Layout | OpCos | Score questions |
|---|---|---|
| UIL Group | UIC, SCG, CNG, CMP | NPS, FCR, CSAT, I_C, C_K, CallReason |
| RGE/NYSEG | RGE, NYSEG | NPS, FCR, CSAT, E_H, C_E, CallReason |

| Source Column (UIL) | Canonical Name |
|---|---|
| `User` | `ExternalDataReference` |
| `Agent` | `Agent` |
| `Date` | `Date` |
| `WorkGroup` | `WorkGroup` |
| `Conversation` | `ConversationID` |
| `Survey` | `SurveyName` |
| `Phone` | `Phone` |
| `Call Reason` | `CallReason` |
| `Status` | `Status` |

| Source Column (RGE/NYSEG) | Canonical Name |
|---|---|
| `ID` | `ExternalDataReference` |
| `Name` | `Agent` |
| `Date/Time` | `Date` |
| `Work Group` | `WorkGroup` |
| `Interaction ID` | `ConversationID` |
| `Survey Name` | `SurveyName` |
| `Phone Number` | `Phone` |
| `Survey Completion` | `Status` |

In [ ]:
# ============================================================
# COLUMN RENAME MAPS
# ============================================================

# UIL group layout  (UIC / SCG / CNG / CMP)
UIL_RENAME = {
    "User"        : "ExternalDataReference",
    "Agent"       : "Agent",
    "Date"        : "Date",
    "WorkGroup"   : "WorkGroup",
    "Conversation": "ConversationID",
    "Survey"      : "SurveyName",
    "Phone"       : "Phone",
    "NPS"         : "NPS",
    "FCR"         : "FCR",
    "CSAT"        : "CSAT",
    "I_C"         : "I_C",
    "C_K"         : "C_K",
    "Call Reason" : "CallReason",
    "Status"      : "Status",
    "Tag"         : "Tag",
}

# RGE / NYSEG layout
RGE_NYSEG_RENAME = {
    "ID"                : "ExternalDataReference",
    "Name"              : "Agent",
    "Date/Time"         : "Date",
    "Work Group"        : "WorkGroup",
    "Interaction ID"    : "ConversationID",
    "Survey Name"       : "SurveyName",
    "Phone Number"      : "Phone",
    "NPS"               : "NPS",
    "FCR"               : "FCR",
    "CSAT"              : "CSAT",
    "E_H"               : "E_H",
    "C_E"               : "C_E",
    "CallReason"        : "CallReason",
    "Survey Completion" : "Status",
    "Tag"               : "Tag",
}

LAYOUT_RENAME = {
    "uil"      : UIL_RENAME,
    "rge_nyseg": RGE_NYSEG_RENAME,
}

# Numeric score columns per layout
SCORE_COLS = {
    "uil"      : ["NPS", "FCR", "CSAT", "I_C", "C_K", "CallReason"],
    "rge_nyseg": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
}


def transform(df, opco):
    """Rename columns and cast score columns to numeric."""
    layout = OPCO_CONFIG[opco]["layout"]
    rename = LAYOUT_RENAME[layout]

    df = df.copy()
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    for col in SCORE_COLS[layout]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    print(f"  [{opco}] Transform complete | Shape: {df.shape}")
    return df

## Validate & Tag

**What it does:**
1. Raises `ValueError` if required score columns are missing
2. Drops fully-empty columns
3. Tags rows where WorkGroup contains `"test"` (case-insensitive)

| Layout | Required score columns |
|---|---|
| UIL Group | NPS, FCR, CSAT, I_C, C_K, CallReason |
| RGE/NYSEG | NPS, FCR, CSAT, E_H, C_E, CallReason |

In [ ]:
# ============================================================
# VALIDATE & TAG
# ============================================================

REQUIRED_SCORE_COLS = {
    "uil"      : ["NPS", "FCR", "CSAT", "I_C", "C_K", "CallReason"],
    "rge_nyseg": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
}


def validate_and_tag(df, opco):
    """Validate required columns, drop empty columns, tag test rows."""
    layout = OPCO_CONFIG[opco]["layout"]
    df = df.copy()

    # Step 1 — Validate required scoring columns
    required = REQUIRED_SCORE_COLS[layout]
    missing  = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"[{opco}] Missing required scoring columns: {missing}")

    # Step 2 — Drop fully-empty columns
    df = df.dropna(axis=1, how="all")

    # Step 3 — Tag test rows by WorkGroup
    if "WorkGroup" in df.columns:
        df["Tag"] = df["WorkGroup"].str.contains("test", case=False, na=False).map(
            {True: "Test", False: ""}
        )
    else:
        df["Tag"] = ""

    print(f"  [{opco}] Validate & tag complete | Shape: {df.shape}")
    if df["Tag"].eq("Test").any():
        print(f"  [{opco}] ⚠  Test rows found: {df['Tag'].eq('Test').sum()}")

    return df

## Filter — Completed Responses Only

Keeps only rows where the `Status` column equals `"completed"` (case-insensitive).
Also drops rows with no ConversationID as a safety net.

| OpCo | Status column | Completion value |
|---|---|---|
| UIL Group | `Status` | `completed` |
| RGE/NYSEG | `Status` | `completed` |

In [ ]:
# ============================================================
# FILTER — Completed Responses Only
# ============================================================

def filter_completed(df, opco):
    """Keep only completed survey responses."""
    total = len(df)

    if "Status" in df.columns:
        df = df[df["Status"].str.strip().str.lower() == "completed"].copy()

    if "ConversationID" in df.columns:
        df = df[
            df["ConversationID"].notna() &
            (df["ConversationID"].astype(str).str.strip() != "")
        ].copy()

    df = df.reset_index(drop=True)
    completed = len(df)

    print(f"  [{opco}] Original records  : {total}")
    print(f"  [{opco}] Completed records : {completed}")
    print(f"  [{opco}] Excluded          : {total - completed}")
    return df

## Prepare for Qualtrics — Field Mapping

**What it does:**
1. Calls the Qualtrics Survey API to retrieve official `questionName`, `questionText`, and QIDs
2. Fuzzy-matches each DataFrame column to Qualtrics `questionName` (cutoff: 0.6)
3. Renames columns to their official Qualtrics names
4. Builds the 3-row header block required by the Qualtrics Import Responses API:
   - Row 1: `questionName` — CSV column headers
   - Row 2: `questionText` — human-readable labels
   - Row 3: `{"ImportId": "QIDx_TEXT"}` — Qualtrics internal mapping
5. Stacks headers + data into the final upload-ready DataFrame

In [ ]:
def prepare_for_qualtrics(df, opco):
    survey_id = SURVEYS_ID[opco]
    url       = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{survey_id}"
    headers   = {"X-API-TOKEN": API_TOKEN}

    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()

    label_map       = response.json()["result"]["questions"]
    qid_to_name     = {qid: q["questionName"] for qid, q in label_map.items()}
    qid_to_text     = {qid: q["questionText"] for qid, q in label_map.items()}
    name_to_qid     = {name: qid for qid, name in qid_to_name.items()}
    qualtrics_names = list(name_to_qid.keys())

    # Step 1 — Fuzzy match columns to Qualtrics questionName
    mapped_cols = {}
    used_names  = set()
    for col in df.columns:
        match = get_close_matches(col, qualtrics_names, n=1, cutoff=0.6)
        if match:
            new_name = match[0]
            if new_name in used_names:
                new_name = col
            mapped_cols[col] = new_name
            used_names.add(new_name)
        else:
            mapped_cols[col] = col
    df = df.rename(columns=mapped_cols)

    # Step 2 — Build questionText row
    row2 = [
        qid_to_text.get(name_to_qid.get(col), "") if name_to_qid.get(col) else ""
        for col in df.columns
    ]

    # Step 3 — Build ImportId row
    row3 = [
        f'{{"ImportId": "{name_to_qid[col]}_TEXT"}}' if col in name_to_qid else ""
        for col in df.columns
    ]

    # Step 4 — Stack headers + data
    hdr1     = pd.DataFrame([list(df.columns)], columns=df.columns)
    hdr2     = pd.DataFrame([row2],             columns=df.columns)
    hdr3     = pd.DataFrame([row3],             columns=df.columns)
    df_final = pd.concat(
        [hdr1, hdr2, hdr3, df.reset_index(drop=True)],
        ignore_index=True
    )

    # Step 5 — Remove duplicate header row (hdr1 mirrors column names)
    df_final = df_final.iloc[1:].reset_index(drop=True)

    print(f"  [{opco}] Qualtrics prep complete | Shape: {df_final.shape}")
    return df_final

## Upload to Qualtrics

Saves the prepared DataFrame to a staging CSV, then POSTs it to the Import Responses API
and polls until completion.

**API flow:**
1. `POST /import-responses` → get `progressId`
2. Poll every 3s until `status == "complete"` or `"failed"`

> A `200 OK` on the POST only means the job was *accepted* — polling confirms actual completion.

**Staging file:** `{OPCO}_Qualtrics_Upload.csv` — overwritten each run (not an audit file).
The audit copy is the `_Qualtrics.csv` file saved alongside the source CSV in the OpCo folder.

In [ ]:
def save_prepared(df_prepared, opco, date_str):
    """Save the Qualtrics-ready DataFrame as a dated audit CSV alongside the source file."""
    cfg    = OPCO_CONFIG[opco]
    folder = cfg["folder"]
    prefix = cfg["file_prefix"]
    out    = os.path.join(folder, f"{prefix} {date_str}_Qualtrics.csv")
    df_prepared.to_csv(out, index=False)
    print(f"  [{opco}] 💾 Audit file saved : {os.path.basename(out)}")
    return out


def upload_to_qualtrics(df_prepared, opco, date_str):
    """
    Saves the Qualtrics-ready DataFrame to a staging CSV, uploads it,
    and polls until the import job completes.
    Returns the final poll result dict.
    """
    survey_id = SURVEYS_ID[opco]
    cfg       = OPCO_CONFIG[opco]

    # Save staging file
    staging_path = os.path.join(cfg["folder"], f"{opco}_Qualtrics_Upload.csv")
    df_prepared.to_csv(staging_path, index=False)
    print(f"  [{opco}] Staging file      : {os.path.basename(staging_path)}")

    url     = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{survey_id}/import-responses"
    headers = {"X-API-TOKEN": API_TOKEN, "Content-Type": "text/csv", "charset": "UTF-8"}

    with open(staging_path, "rb") as f:
        resp = requests.post(url, headers=headers, data=f, verify=False)
    resp.raise_for_status()

    progress_id = resp.json()["result"]["progressId"]
    print(f"  [{opco}] Job accepted      | progressId: {progress_id}")

    poll_url     = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{survey_id}/import-responses/{progress_id}"
    poll_headers = {"X-API-TOKEN": API_TOKEN}

    for attempt in range(30):
        time.sleep(3)
        poll_result = requests.get(poll_url, headers=poll_headers, verify=False).json()["result"]
        pct         = poll_result.get("percentComplete", 0)
        status      = poll_result.get("status", "unknown")
        print(f"    [{attempt + 1:02d}] {status} | {pct:.0f}%")

        if status == "complete":
            print(f"  [{opco}] ✅ Upload confirmed complete.")
            return poll_result
        if status == "failed":
            raise RuntimeError(f"Qualtrics import failed: {poll_result}")

    raise TimeoutError("Upload did not complete within 90 seconds.")

---
# 3. Execute

Runs the full ETL pipeline end-to-end for all OpCos that have a file loaded.

**Pipeline per OpCo:**
```
1. Duplicate check  → query Qualtrics for latest Date already ingested
2. Transform        → rename columns, cast score columns to numeric
3. Validate & tag   → validate columns, tag test rows
4. Filter           → keep only completed responses
5. Prepare          → map to Qualtrics field names + build 3-row header
6. Save audit CSV   → <prefix> YYYY-MM-DD_Qualtrics.csv (dated, kept for audit)
7. Upload           → POST + poll until complete
```

**File output per OpCo per run:**
```
Alorica_RGE/
    ALO RGE_Survey_Data 2026-10-13.csv              ← source file (placed manually)
    ALO RGE_Survey_Data 2026-10-13_Qualtrics.csv    ← upload-ready audit copy
    RGE_Qualtrics_Upload.csv                        ← staging file (overwritten each run)

Alorica_UIL/
    ALO CMP_Survey_Data 2026-10-13.csv
    ALO CMP_Survey_Data 2026-10-13_Qualtrics.csv
    CMP_Qualtrics_Upload.csv
    (same pattern for SCG, CNG, UIC)
```

In [ ]:
# ============================================================
# EXECUTE — Full ETL Pipeline for all loaded OpCos
# ============================================================
# Set DRY_RUN = True to prep and save audit files without uploading.

DRY_RUN = False

print("=" * 60)
print(f"  Alorica → Qualtrics Ingestion")
print(f"  Date    : {PIPELINE_DATE}")
print(f"  Upload  : {'DRY RUN — no upload' if DRY_RUN else 'YES'}")
print("=" * 60)

RESULTS = {}

for opco in OPCO_CONFIG:
    print(f"\n▶ {opco}")

    df = RAW_DATA.get(opco)
    if df is None:
        print(f"  [{opco}] Skipping — no data loaded")
        RESULTS[opco] = "FILE_NOT_FOUND"
        continue

    try:
        # Step 1 — Duplicate check against Qualtrics
        print(f"  [{opco}] Checking Qualtrics for existing data...")
        qualtrics_check = check_latest_qualtrics_date(opco)

        if qualtrics_check is not None:
            latest_date, latest_recorded, batch_count = qualtrics_check
            qualtrics_date = latest_date.date()
            pipeline_date  = date.fromisoformat(PIPELINE_DATE)

            print(f"  [{opco}] Qualtrics latest date : {qualtrics_date}")
            print(f"  [{opco}] Pipeline date         : {pipeline_date}")

            if qualtrics_date >= pipeline_date:
                print(f"  [{opco}] ⚠  Duplicate detected — skipping (already ingested {batch_count} records for {qualtrics_date})")
                RESULTS[opco] = "DUPLICATE_SKIPPED"
                continue

        # Step 2 — Transform
        df = transform(df, opco)

        # Step 3 — Validate & tag
        df = validate_and_tag(df, opco)

        # Step 4 — Filter completed responses
        df_completed = filter_completed(df, opco)

        if df_completed.empty:
            print(f"  [{opco}] Skipping — no completed responses after filter")
            RESULTS[opco] = "NO_DATA"
            continue

        # Step 5 — Prepare Qualtrics-ready format
        df_q = prepare_for_qualtrics(df_completed, opco)

        # Step 6 — Save dated audit CSV
        try:
            save_prepared(df_q, opco, PIPELINE_DATE)
        except Exception as e:
            print(f"  [{opco}] ⚠  Could not save audit file: {e}")

        # Step 7 — Upload
        if DRY_RUN:
            print(f"  [{opco}] DRY RUN — upload skipped")
            RESULTS[opco] = "DRY_RUN_OK"
        else:
            upload_to_qualtrics(df_q, opco, PIPELINE_DATE)
            RESULTS[opco] = "UPLOADED"

    except Exception as e:
        print(f"  [{opco}] ❌ Pipeline failed: {e}")
        RESULTS[opco] = f"ERROR: {e}"

# ── Summary ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY")
print("=" * 60)
for opco, status in RESULTS.items():
    icon = "✅" if status in ("UPLOADED", "DRY_RUN_OK") else "⚠️ "
    print(f"  {icon} {opco:8} : {status}")
print("=" * 60)